# RetinaGuard — Train, Evaluate & Auto-Publish on APTOS 2019 (Kaggle)

Runs the full pipeline, then commits the proof artifacts to GitHub and publishes the trained weights as a GitHub Release.

## Setup before running (right sidebar / Add-ons)
1. **Accelerator → GPU** (T4 or P100).
2. **Internet → ON** (needed for git clone, pip, GitHub API).
3. **Input → + Add Input → Competitions → `aptos2019-blindness-detection` → Add.** (Join once: https://www.kaggle.com/c/aptos2019-blindness-detection/rules )
4. **Add-ons → Secrets → Add a secret** named **`GITHUB_TOKEN`** = a fine-grained GitHub PAT (repo: RetinaGuard, Contents: read/write). Tick it for this notebook.

## IMPORTANT — if you hit CUDA out of memory
It's almost always orphaned GPU memory from an earlier failed run. Fix: **Run → Factory reset**, then run cells fresh. Cell 1 prints GPU memory so you can confirm it starts near 0.

## To run overnight unattended (recommended)
**Save Version → Save & Run All (Commit)** runs headless on Kaggle (GPU, up to ~9h) — laptop can be off. Always gets a fresh GPU, so no orphaned-memory issues. The GitHub commit + Release happen automatically at the end.

Run cells top to bottom (do NOT merge them).

## 1. Confirm GPU (and that it's empty) + auto-locate data + config knobs

In [ ]:
import torch, os, glob
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
print('^ memory.used should be near 0. If it is multiple GB, do Run -> Factory reset before training.')

# Auto-detect the APTOS files wherever Kaggle mounted them under /kaggle/input
_csv = glob.glob('/kaggle/input/**/train.csv', recursive=True)
_img = glob.glob('/kaggle/input/**/train_images', recursive=True)
assert _csv, 'train.csv not found under /kaggle/input — is the APTOS competition attached?'
assert _img, 'train_images/ not found under /kaggle/input — is the APTOS competition attached?'
CSV_PATH   = _csv[0]
IMAGES_DIR = _img[0]
print('CSV_PATH   =', CSV_PATH)
print('IMAGES_DIR =', IMAGES_DIR)
print('images:', len(glob.glob(IMAGES_DIR + '/*')), '| csv exists:', os.path.exists(CSV_PATH))

# Config knobs. 380px + BS=8 fits a CLEAN 14.5GB T4. If you still OOM after a Factory reset,
# step down: IMG=320/BS=6, then IMG=288/BS=4.
IMG    = 380
BS     = 8
EPOCHS = 60
GITHUB_REPO = 'Gaurav-0704/RetinaGuard'

## 2. Clone the repo + patch config + safety-patch torch.load (verify all landed)

In [ ]:
import re
%cd /kaggle/working
![ -d RetinaGuard ] && rm -rf RetinaGuard
!git clone https://github.com/Gaurav-0704/RetinaGuard.git
%cd RetinaGuard
cfg = open('ml/config.py').read()
cfg = re.sub(r'^IMAGE_SIZE.*$', f'IMAGE_SIZE        = {IMG}', cfg, flags=re.M)
cfg = re.sub(r'^BATCH_SIZE.*$',  f'BATCH_SIZE        = {BS}',  cfg, flags=re.M)
cfg = re.sub(r'^NUM_EPOCHS.*$',  f'NUM_EPOCHS        = {EPOCHS}', cfg, flags=re.M)
open('ml/config.py','w').write(cfg)
print(f'Requested -> IMG={IMG} BS={BS} EPOCHS={EPOCHS}')
print('Actually written to ml/config.py:')
!grep -E '^(IMAGE_SIZE|BATCH_SIZE|NUM_EPOCHS)' ml/config.py

# Safety patch: PyTorch 2.6+ defaults torch.load(weights_only=True), which rejects our
# checkpoint's numpy scalar metadata. Force weights_only=False (idempotent — no-op if already fixed).
!sed -i 's/torch.load(weights_path, map_location=device)/torch.load(weights_path, map_location=device, weights_only=False)/' ml/model.py
!grep -n 'torch.load' ml/model.py

!python -c "import ast; ast.parse(open('ml/train.py').read()); ast.parse(open('ml/evaluate.py').read()); ast.parse(open('ml/model.py').read()); print('train/evaluate/model parse OK')"

## 3. Ensure deps (Kaggle already ships torch/sklearn/pandas/matplotlib)

In [ ]:
!pip -q install opencv-python-headless 2>/dev/null
import cv2, sklearn; print('cv2', cv2.__version__, '| sklearn', sklearn.__version__)

## 4. Smoke test — 2 epochs (confirms train + evaluate end-to-end)

In [ ]:
!cp ml/config.py /tmp/config_full.py
!sed -i 's/^NUM_EPOCHS.*/NUM_EPOCHS        = 2/' ml/config.py
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m ml.train --images_dir "$IMAGES_DIR" --csv_path "$CSV_PATH"
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m ml.evaluate --images_dir "$IMAGES_DIR" --csv_path backend/weights/test_split.csv
!cp /tmp/config_full.py ml/config.py
print('Smoke test done — a classification report + metrics above means you are good to go.')

## 5. Full training run (~30-60 min)

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m ml.train --images_dir "$IMAGES_DIR" --csv_path "$CSV_PATH"

## 6. Evaluate on the held-out test split

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m ml.evaluate --images_dir "$IMAGES_DIR" --csv_path backend/weights/test_split.csv

## 7. Show the results inline + copy to /kaggle/working (Output tab)

In [ ]:
import shutil, os
print(open('docs/results/report.md').read())
from IPython.display import Image, display
for p in ['docs/results/confusion_matrix.png','docs/results/confusion_matrix_normalized.png','docs/results/roc_curves.png']:
    display(Image(p))
os.makedirs('/kaggle/working/outputs', exist_ok=True)
for src in ['backend/weights/retinoguard_best.pth','backend/weights/history.json','backend/weights/test_split.csv']:
    if os.path.exists(src): shutil.copy(src, '/kaggle/working/outputs/')
if os.path.isdir('docs/results'):
    shutil.copytree('docs/results', '/kaggle/working/outputs/results', dirs_exist_ok=True)
print('Saved to /kaggle/working/outputs:', os.listdir('/kaggle/working/outputs'))

## 8. Commit proof artifacts to GitHub + publish weights as a Release
Requires the `GITHUB_TOKEN` secret (step 4 at top). Commits the small proofs to `docs/results/`; creates a Release and uploads `retinoguard_best.pth`. No large files enter git history.

In [ ]:
import os, json, shutil, datetime, requests
from kaggle_secrets import UserSecretsClient

TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert TOKEN, 'Add a Kaggle Secret named GITHUB_TOKEN (fine-grained PAT, Contents: read/write).'

# 1) Stage small proof artifacts inside the repo (never the weights)
os.makedirs('docs/results', exist_ok=True)
for src in ['backend/weights/history.json', 'backend/weights/test_split.csv']:
    if os.path.exists(src): shutil.copy(src, 'docs/results/')
metrics = json.load(open('docs/results/metrics.json'))
print('Metrics to publish:', metrics['overall'])

# 2) Commit + push proofs (token-authenticated remote)
!git config user.email "gauravsinght007@gmail.com"
!git config user.name "Gaurav-0704"
!git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git
!git add docs/results docs/RESULTS.md
!git commit -m "Add training proof artifacts (metrics, confusion matrix, ROC, history, held-out split)" || echo "nothing to commit"
!git push origin HEAD:main

# 3) Create a GitHub Release and upload the .pth
H = {'Authorization': f'token {TOKEN}', 'Accept': 'application/vnd.github+json'}
tag = 'model-v1-' + datetime.datetime.now().strftime('%Y%m%d-%H%M')
o = metrics['overall']
body = (f"RetinaGuard EfficientNet-B4 trained on APTOS 2019 (held-out test set: "
        f"{metrics['n_test_samples']} images).\n\n"
        f"- Accuracy: {o['accuracy']}\n"
        f"- Quadratic Weighted Kappa: {o['quadratic_weighted_kappa']}\n"
        f"- Macro F1: {o['f1_macro']}\n"
        f"- Macro AUC (OvR): {o['macro_auc_ovr']}\n\n"
        f"Full per-class metrics, confusion matrix and ROC curves are in docs/results/.")
r = requests.post(f'https://api.github.com/repos/{GITHUB_REPO}/releases', headers=H,
                  json={'tag_name': tag, 'name': f'Trained model {tag}', 'body': body})
r.raise_for_status()
rel = r.json(); upload_url = rel['upload_url'].split('{')[0]
print('Release created:', rel['html_url'])

pth = os.path.realpath('backend/weights/retinoguard_best.pth')
assert os.path.exists(pth), 'No trained checkpoint found to upload.'
with open(pth, 'rb') as fh:
    up = requests.post(upload_url + '?name=retinoguard_best.pth',
                       headers={**H, 'Content-Type': 'application/octet-stream'}, data=fh)
up.raise_for_status()
print('Weights uploaded:', up.json()['browser_download_url'])
print('\nDONE. Proofs committed to docs/results/, weights attached to the Release above.')

## 9. Sanity check
Strong single-model APTOS results land around QWK 0.88-0.91 / accuracy ~0.82-0.85. QWK > 0.95 suggests a data leak — investigate before reporting.